In [1]:
import os

base = r"C:\Users\ACER NITRO\Documents\IPCV\Assignment_IPCV_Knee\Knee_Dataset"

for split in ["train", "val", "test", "auto_test"]:
    split_path = os.path.join(base, split)
    print(f"\n{split}")
    total = 0
    for cls in sorted(os.listdir(split_path)):
        n = len(os.listdir(os.path.join(split_path, cls)))
        total += n
        print(f"  grade {cls}: {n}")
    print(f"  total: {total}")


train
  grade 0: 2286
  grade 1: 1046
  grade 2: 1516
  grade 3: 757
  grade 4: 173
  total: 5778

val
  grade 0: 328
  grade 1: 153
  grade 2: 212
  grade 3: 106
  grade 4: 27
  total: 826

test
  grade 0: 639
  grade 1: 296
  grade 2: 447
  grade 3: 223
  grade 4: 51
  total: 1656

auto_test
  grade 0: 604
  grade 1: 275
  grade 2: 403
  grade 3: 200
  grade 4: 44
  total: 1526


In [2]:
import os
import numpy as np
import cv2
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

BASE = r"C:\Users\ACER NITRO\Documents\IPCV\Assignment_IPCV_Knee\Knee_Dataset"

def load_split(split):
    """Read every image in a split and return HOG features + labels."""
    X, y = [], []
    for grade in sorted(os.listdir(os.path.join(BASE, split))):
        folder = os.path.join(BASE, split, grade)
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (128, 128))
            X.append(hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2)))
            y.append(int(grade))
    return np.array(X), np.array(y)

X_train, y_train = load_split("train")
X_test,  y_test  = load_split("test")

# Scale features: fit on train only, apply the same scaling to test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Majority-class baseline: accuracy of always predicting the biggest class (grade 0)
majority = np.bincount(y_train).argmax()
baseline_acc = np.mean(y_test == majority)
print(f"Majority baseline accuracy: {baseline_acc:.3f}")

# Train the model on the imbalanced data (no class weights yet)
model = SVC(kernel="rbf")
model.fit(X_train, y_train)

# Compare train vs test accuracy (the gap shows overfitting)
print(f"Train accuracy: {accuracy_score(y_train, model.predict(X_train)):.3f}")
y_pred = model.predict(X_test)
print(f"Test accuracy : {accuracy_score(y_test, y_pred):.3f}")

# Per-class results: look at recall for each grade
print(classification_report(y_test, y_pred, digits=3))

Majority baseline accuracy: 0.386
Train accuracy: 0.778
Test accuracy : 0.529
              precision    recall  f1-score   support

           0      0.551     0.936     0.694       639
           1      1.000     0.003     0.007       296
           2      0.492     0.490     0.491       447
           3      0.449     0.238     0.311       223
           4      0.714     0.098     0.172        51

    accuracy                          0.529      1656
   macro avg      0.641     0.353     0.335      1656
weighted avg      0.607     0.529     0.449      1656



In [3]:
import os
import numpy as np
import cv2
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from imblearn.over_sampling import RandomOverSampler, SMOTE

BASE = r"C:\Users\ACER NITRO\Documents\IPCV\Assignment_IPCV_Knee\Knee_Dataset"

def load_split(split):
    X, y = [], []
    for grade in sorted(os.listdir(os.path.join(BASE, split))):
        folder = os.path.join(BASE, split, grade)
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (128, 128))
            X.append(hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2)))
            y.append(int(grade))
    return np.array(X), np.array(y)

# Load once and cache to disk, so the next run skips the slow image loading
if os.path.exists("features.npz"):
    d = np.load("features.npz")
    X_train, y_train, X_test, y_test = d["Xtr"], d["ytr"], d["Xte"], d["yte"]
else:
    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")
    np.savez("features.npz", Xtr=X_train, ytr=y_train, Xte=X_test, yte=y_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

def evaluate(name, model, X_tr, y_tr):
    """Train on (X_tr, y_tr), always evaluate on the untouched test set."""
    model.fit(X_tr, y_tr)
    pred = model.predict(X_test)
    recalls = recall_score(y_test, pred, average=None)  # one recall per grade
    return {
        "Method": name,
        "Train acc": accuracy_score(y_tr, model.predict(X_tr)),
        "Test acc": accuracy_score(y_test, pred),
        "Macro-F1": f1_score(y_test, pred, average="macro"),
        "Recall G1": recalls[1],
        "Recall G3": recalls[3],
        "Recall G4": recalls[4],
    }

results = []

# 1. Baseline (from before)
results.append(evaluate("Baseline", SVC(kernel="rbf"), X_train, y_train))

# 2. Class weights: penalise mistakes on rare classes more
results.append(evaluate("Class weight", SVC(kernel="rbf", class_weight="balanced"),
                        X_train, y_train))

# 3. Random oversampling: duplicate minority images until classes are equal
X_ros, y_ros = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)
results.append(evaluate("Random oversampling", SVC(kernel="rbf"), X_ros, y_ros))

# 4. SMOTE: create new synthetic minority samples between existing neighbours
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
results.append(evaluate("SMOTE", SVC(kernel="rbf"), X_sm, y_sm))

# Summary table
print(f"{'Method':<22}{'Train':>7}{'Test':>7}{'MacroF1':>9}{'R-G1':>7}{'R-G3':>7}{'R-G4':>7}")
for r in results:
    print(f"{r['Method']:<22}{r['Train acc']:>7.3f}{r['Test acc']:>7.3f}"
          f"{r['Macro-F1']:>9.3f}{r['Recall G1']:>7.3f}{r['Recall G3']:>7.3f}{r['Recall G4']:>7.3f}")

Method                  Train   Test  MacroF1   R-G1   R-G3   R-G4
Baseline                0.778  0.529    0.335  0.003  0.238  0.098
Class weight            0.838  0.478    0.462  0.284  0.498  0.392
Random oversampling     0.960  0.499    0.447  0.189  0.417  0.373
SMOTE                   0.967  0.507    0.429  0.149  0.363  0.314
